# İstanbul Barajları Doluluk Analizi

**Veri seti:** İBB Açık Veri Portalı – *İstanbul Barajları Günlük Doluluk Oranları* (`istanbul-dams-daily-occupancy-rates.xlsx`), 10 baraj, günlük, Ekim 2000 – Şubat 2024.

**Önerme:** *"İstanbul barajları her yıl en düşük seviyesine sonbaharda (Eylül–Aralık) iner."*

Kullanılan araçlar: yalnızca Python, Pandas ve NumPy.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("istanbul-dams-daily-occupancy-rates.xlsx")
print(df.shape)
print(df.dtypes)
print(df.isna().sum())

(8520, 11)
Tarih            object
Ömerli          float64
Darlık          float64
Elmalı          float64
Terkos          float64
Alibey          float64
Büyükçekmece    float64
Sazlıdere       float64
Kazandere       float64
Pabuçdere       float64
Istrancalar     float64
dtype: object
Tarih           0
Ömerli          0
Darlık          0
Elmalı          0
Terkos          0
Alibey          0
Büyükçekmece    0
Sazlıdere       0
Kazandere       0
Pabuçdere       0
Istrancalar     0
dtype: int64


## 1. Veri kalitesi kontrolü

Eksik değer yok, ama iki sorun var:
1. `Tarih` kolonu karışık tipte: çoğu satır datetime, bir kısmı metin (`"1.4.2023"` gibi).
2. Doluluk oranları 0–1 arasında olmalı, fakat bazı değerler 15–80 arası. Bu satırlar **yüzde** olarak girilmiş.

In [2]:
baraj_kolonlari = ["Ömerli", "Darlık", "Elmalı", "Terkos", "Alibey",
                   "Büyükçekmece", "Sazlıdere", "Kazandere", "Pabuçdere", "Istrancalar"]

# Tarih kolonundaki tip dağılımı
print(df["Tarih"].map(type).value_counts())

# Metin olarak girilmiş tarihler
metin_tarih = df["Tarih"].map(lambda x: isinstance(x, str))
print("Metin tarih aralığı:", df.loc[metin_tarih, "Tarih"].iloc[0], "->", df.loc[metin_tarih, "Tarih"].iloc[-1])

# Yüzde ölçeğinde girilmiş satırlar (1.5'ten büyük değerler oran olamaz)
yuzde_satir = (df[baraj_kolonlari] > 1.5).any(axis=1)
print("Yüzde ölçeğindeki satır sayısı:", yuzde_satir.sum())
print("Bu satırlar metin tarihli satırlarla aynı mı?", (yuzde_satir == metin_tarih).all())

Tarih
<class 'datetime.datetime'>    8390
<class 'str'>                   130
Name: count, dtype: int64
Metin tarih aralığı: 1.4.2023 -> 8.8.2023
Yüzde ölçeğindeki satır sayısı: 130
Bu satırlar metin tarihli satırlarla aynı mı? True


Hatalı satırların hepsi aynı dönemde (1 Nisan – 8 Ağustos 2023): o dönemin verisi hem farklı tarih formatıyla hem de yüzde cinsinden girilmiş.

Sazlıdere ve Pabuçdere'de 1'i *hafifçe* aşan (1.00–1.10) değerler de var. Bunlar barajın tam kapasitenin üstüne çıkması (taşma) olarak gerçekçi, bu yüzden onlara dokunmuyoruz. Eşik olarak 1.5 seçmemizin nedeni bu.

In [3]:
# Düzeltme 1: metin tarihleri gün.ay.yıl formatıyla çevir
df.loc[metin_tarih, "Tarih"] = pd.to_datetime(df.loc[metin_tarih, "Tarih"], format="%d.%m.%Y")
df["Tarih"] = pd.to_datetime(df["Tarih"])

# Düzeltme 2: yüzde ölçeğindeki satırları 100'e böl
df.loc[yuzde_satir, baraj_kolonlari] = df.loc[yuzde_satir, baraj_kolonlari] / 100

df = df.sort_values("Tarih").reset_index(drop=True)
print("Tekrarlanan tarih:", df["Tarih"].duplicated().sum())
print("Max değer düzeltme sonrası:", df[baraj_kolonlari].max().max().round(3))

Tekrarlanan tarih: 0
Max değer düzeltme sonrası: 1.0


## 2. Genel doluluk oranı

10 barajın **ağırlıksız ortalaması** alınıyor. Not: İSKİ'nin resmi genel oranı baraj kapasitelerine göre ağırlıklıdır; veri setinde kapasite olmadığı için burada düz ortalama kullanıldı, bu yüzden değerler resmi rakamlardan farklı olabilir.

In [4]:
RATE = "GENEL_DOLULUK_ORANI"
df[RATE] = df[baraj_kolonlari].mean(axis=1)
df["yil"] = df["Tarih"].dt.year
df["ay"] = df["Tarih"].dt.month

print(df[RATE].describe().round(3))

count    8520.000
mean        0.625
std         0.238
min         0.105
25%         0.425
50%         0.650
75%         0.834
max         0.998
Name: GENEL_DOLULUK_ORANI, dtype: float64


## 3. Keşifsel analiz (EDA)

In [5]:
# Yıllık özet
yillik = df.groupby("yil")[RATE].agg(["count", "mean", "min", "max"]).round(3)
print(yillik)

      count   mean    min    max
yil                             
2000     70  0.420  0.403  0.442
2001    365  0.515  0.293  0.838
2002    365  0.758  0.586  0.917
2003    365  0.827  0.665  0.998
2004    366  0.772  0.397  0.977
2005    365  0.687  0.378  0.945
2006    365  0.810  0.577  0.994
2007    365  0.347  0.110  0.616
2008    366  0.284  0.105  0.443
2009    365  0.815  0.366  0.981
2010    365  0.860  0.678  0.995
2011    365  0.806  0.551  0.982
2012    366  0.712  0.428  0.949
2013    365  0.635  0.295  0.931
2014    365  0.300  0.144  0.707
2015    365  0.807  0.595  0.976
2016    366  0.589  0.288  0.865
2017    365  0.661  0.396  0.831
2018    365  0.714  0.442  0.897
2019    365  0.679  0.305  0.955
2020    366  0.412  0.197  0.647
2021    365  0.505  0.193  0.774
2022    365  0.624  0.285  0.913
2023    365  0.286  0.133  0.558
2024     50  0.622  0.507  0.738


In [6]:
# Baraj bazında ortalama doluluk ve oynaklık (std)
baraj_ozet = pd.DataFrame({
    "ortalama": df[baraj_kolonlari].mean(),
    "std": df[baraj_kolonlari].std(),
    "min": df[baraj_kolonlari].min(),
}).round(3).sort_values("ortalama", ascending=False)
print(baraj_ozet)

              ortalama    std    min
Terkos           0.704  0.210  0.077
Ömerli           0.694  0.234  0.051
Darlık           0.676  0.245  0.054
Elmalı           0.660  0.286  0.005
Istrancalar      0.624  0.315  0.055
Kazandere        0.623  0.362  0.005
Büyükçekmece     0.612  0.263  0.016
Pabuçdere        0.564  0.367  0.000
Sazlıdere        0.551  0.268  0.012
Alibey           0.538  0.290  0.000


In [7]:
# En kurak ve en dolu yıllar (yalnızca 12 ayı tam olan yıllar)
tam_yillar = df.groupby("yil").filter(lambda g: g["ay"].nunique() == 12)
yil_ort = tam_yillar.groupby("yil")[RATE].mean()
print("En kurak 3 yıl:\n", yil_ort.nsmallest(3).round(3))
print("En dolu 3 yıl:\n", yil_ort.nlargest(3).round(3))

En kurak 3 yıl:
 yil
2008    0.284
2023    0.286
2014    0.300
Name: GENEL_DOLULUK_ORANI, dtype: float64
En dolu 3 yıl:
 yil
2010    0.860
2003    0.827
2009    0.815
Name: GENEL_DOLULUK_ORANI, dtype: float64


In [8]:
# Aylara göre ortalama doluluk (mevsimsellik)
aylik = df.groupby("ay")[RATE].mean().round(3)
print(aylik)
print("En düşük ay:", aylik.idxmin(), "| En yüksek ay:", aylik.idxmax())

ay
1     0.621
2     0.717
3     0.779
4     0.797
5     0.759
6     0.701
7     0.627
8     0.548
9     0.489
10    0.472
11    0.475
12    0.526
Name: GENEL_DOLULUK_ORANI, dtype: float64
En düşük ay: 10 | En yüksek ay: 4


## 4. Önerme testi

Her tam yıl için doluluğun en düşük olduğu günü buluyoruz ve bu günün hangi ayda olduğuna bakıyoruz.

`tam_yillar` kullanılmasının nedeni: veri Ekim 2000'de başlıyor, Şubat 2024'te bitiyor. Bu eksik yıllarda dip noktası sadece mevcut aylar içinden seçilir ve sonucu yanıltır (örneğin 2024'ün "dibi" mecburen Ocak veya Şubat çıkar).

In [9]:
dip = tam_yillar.loc[tam_yillar.groupby("yil")[RATE].idxmin(), ["yil", "Tarih", RATE]]
dip["dip_ay"] = dip["Tarih"].dt.month
print(dip.to_string(index=False))
print()
print(dip["dip_ay"].value_counts().sort_index())

 yil      Tarih  GENEL_DOLULUK_ORANI  dip_ay
2001 2001-11-19             0.293476      11
2002 2002-08-18             0.585757       8
2003 2003-10-16             0.664765      10
2004 2004-12-21             0.396533      12
2005 2005-01-15             0.378077       1
2006 2006-09-22             0.576688       9
2007 2007-11-05             0.110445      11
2008 2008-09-25             0.105332       9
2009 2009-01-02             0.365919       1
2010 2010-10-07             0.678263      10
2011 2011-10-09             0.550880      10
2012 2012-10-20             0.428290      10
2013 2013-12-31             0.294570      12
2014 2014-09-16             0.144250       9
2015 2015-12-31             0.595090      12
2016 2016-11-27             0.287820      11
2017 2017-01-04             0.396450       1
2018 2018-11-15             0.441970      11
2019 2019-12-27             0.304620      12
2020 2020-12-31             0.196940      12
2021 2021-01-09             0.192850       1
2022 2022-

In [10]:
sonbahar = np.isin(dip["dip_ay"], [9, 10, 11, 12])
genis = np.isin(dip["dip_ay"], [9, 10, 11, 12, 1])

print(f"Toplam tam yıl: {len(dip)}")
print(f"Dip Eylül–Aralık arasında: {sonbahar.sum()} yıl (%{sonbahar.mean()*100:.1f})")
print(f"Dip Eylül–Ocak arasında:   {genis.sum()} yıl (%{genis.mean()*100:.1f})")
print("İstisna yıllar:", dip.loc[~genis, ["yil", "dip_ay"]].values.tolist())

Toplam tam yıl: 23
Dip Eylül–Aralık arasında: 18 yıl (%78.3)
Dip Eylül–Ocak arasında:   22 yıl (%95.7)
İstisna yıllar: [[2002, 8]]


## 5. Sonuç

**Önerme büyük ölçüde destekleniyor.**

- Aylık ortalamalarda belirgin bir mevsimsellik var: doluluk Nisan'da zirve yapıyor (~%80), Ekim'de dibe iniyor (~%47). Kış ve bahar yağışlarıyla dolan barajlar yaz boyunca tüketimle boşalıyor.
- 23 tam yılın **18'inde (%78.3)** yılın en düşük seviyesi Eylül–Aralık arasında görülüyor.
- Kalan 5 yılın 4'ünde dip Ocak'ın ilk iki haftasında (2005, 2009, 2017, 2021). Bu, ayrı bir olay değil, sonbahar düşüşünün yıl sınırını aşarak devam etmesi. Bu yıllar da dahil edildiğinde oran **%95.7 (22/23)**.
- Tek gerçek istisna **2002** (dip Ağustos'ta). O yıl doluluk zaten yüksek seyretmiş ve dip seviyesi bile %58 civarında kalmış.

**Ek bulgular:**
- En kurak yıllar 2008, 2023 ve 2014 (yıllık ortalama ~%28–30); en dolu yıllar 2010, 2003 ve 2009.
- Veri setinde 1 Nisan – 8 Ağustos 2023 arasındaki 130 satır hem metin formatında tarihle hem de yüzde ölçeğinde girilmişti. Bu düzeltilmeseydi 2023 ortalaması 12.3 gibi imkânsız bir değer çıkıyordu ve aylık mevsimsellik tablosu bozuluyordu.

**Kısıtlar:** Genel doluluk barajların ağırlıksız ortalaması; kapasiteye göre ağırlıklandırılmış resmi İSKİ oranı farklı olabilir. Analiz yağış verisi içermediği için nedensellik değil, sadece örüntü gösteriyor.